# AlternativeNorm1 Quantization Error Analysis

This notebook visualizes and analyzes the quantization errors computed by **`AlternativeNorm1Analysis`** across a grid of integer bitwidths ($ib$) and fractional bitwidths ($fb$).

### Normalization Equation
The **AlternativeNorm1** normalizer normalizes detector ring energy profiles per event by dividing each ring energy by the absolute value of the sum of ring energies across all rings in that event:

$$\hat{x}_i = \frac{x_i}{\left|\sum_{j=1}^D x_j\right|} \quad \left(\text{or } x_i \text{ if } \left|\sum_{j=1}^D x_j\right| = 0\right)$$

### Visualizations in this Notebook
1. **Aggregated Metrics Heatmap Grid**: 2D heatmaps of MSE, RMSE, MAE, MAPE, Max Error, and KL divergence across $(ib, fb)$.
2. **2D Error Distribution & Marginals**: 2D histogram (`hist2d`) of sample error distributions as a function of fractional bits and integer bits with top and right marginal projections.
3. **Feature-level (Variable) Joint Distributions & Marginals**: 2D histogram of true normalized ring features vs. quantization error ($|y - \hat{y}|$) with marginal distributions for each variable.
4. **Marginal Error Profiles Comparison**: Overlaid 1D marginal distributions comparing error suppression across bitwidths.
5. **Ring-wise Spatial Error Profile**: Absolute quantization error per detector ring index.

In [ ]:
from pathlib import Path
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import mplhep as hep

# Apply ATLAS style
plt.style.use(hep.style.ATLAS)

# Add project root to sys.path
from utils import add_project_root_to_sys_path
add_project_root_to_sys_path()

from neuralnet.workflows.ringer.alternative_norm1 import AlternativeNorm1Analysis, IntegerRange
from neuralnet.normalizers.polars import AlternativeNorm1
from neuralnet.utils.polars import RingSlicesPerLayer
from neuralnet.plotting import (
    plot_metrics_heatmap,
    plot_aggregated_metrics_grid,
    plot_joint_hist2d_with_marginals,
    plot_error_distributions_by_bits,
)
from neuralnet import get_logger

logger = get_logger()

## 1. Setup Configuration & Run / Load Analysis

You can run the analysis on a real Ringer dataset (or generate a synthetic dataset for demonstration if no pre-computed output exists).
The analysis tests integer bit range $[start, stop)$ and fractional bit range $[start, stop)$.

In [ ]:
# Set analysis directory paths
data_dir = Path("../data/ringer_norm1_study").resolve()
output_dir = Path("../jobs/alternative_norm1_study").resolve()
data_dir.mkdir(parents=True, exist_ok=True)
data_file = data_dir / "data.parquet"

# Generate representative ring data if not present
if not data_file.exists():
    rng = np.random.default_rng(42)
    n_samples = 5000
    n_rings = 100
    ring_decay = np.exp(-np.linspace(0, 3, n_rings))
    rings = [
        (rng.exponential(scale=ring_decay) * rng.uniform(1.0, 50.0)).astype(np.float32).tolist()
        for _ in range(n_samples)
    ]
    df_synth = pl.DataFrame({
        "id": np.arange(n_samples, dtype=np.int64),
        "rings": rings,
        "et": rng.uniform(15000.0, 50000.0, n_samples).astype(np.float32),
        "eta": rng.uniform(0.0, 2.5, n_samples).astype(np.float32),
    })
    df_synth.write_parquet(data_file)
    logger.info(f"Saved dataset with {n_samples} samples to {data_file}")

# Configure AlternativeNorm1Analysis
analysis = AlternativeNorm1Analysis(
    dataset_dir=data_dir,
    data_table="data.parquet",
    rings_col="rings",
    ring_fraction=2,
    integer_bits_range=IntegerRange(start=1, stop=4),      # ib in {1, 2, 3}
    fractional_bits_range=IntegerRange(start=4, stop=13),  # fb in {4, ..., 12}
    output_path=output_dir,
)

# Submit analysis or load saved results
if (output_dir / "results.csv").exists():
    logger.info(f"Loading existing results from {output_dir}")
    analysis = AlternativeNorm1Analysis.load(output_dir)
    results_df = analysis.results.collect()
else:
    logger.info("Running AlternativeNorm1Analysis...")
    results_df = analysis.submit()

results_df

## 2. Heatmaps of Aggregated Metrics

We plot a 2D grid of heatmaps for each aggregated metric as a function of integer bits ($ib$) on the y-axis and fractional bits ($fb$) on the x-axis.

In [ ]:
fig, axes = plot_aggregated_metrics_grid(
    data=results_df,
    metrics=["mse", "rmse", "mae", "mape", "max_error", "kl_divergence"],
    ncols=3,
    figsize=(18, 10),
    cmap="viridis_r",
    log_scale=True,
    annot=True,
    fmt=".2e",
)
plt.suptitle("AlternativeNorm1 Quantization Metrics Grid ($ib$ vs $fb$)", fontsize=18, y=1.02)
plt.show()

### Individual Metric Heatmaps
Inspect Mean Absolute Error (MAE) and Spatial KL Divergence ($D_{\mathrm{KL}}$) in detail:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

plot_metrics_heatmap(
    data=results_df,
    metric_col="mae",
    title="Mean Absolute Error (MAE)",
    ax=ax1,
    cmap="magma_r",
    fmt=".2e",
)

plot_metrics_heatmap(
    data=results_df,
    metric_col="kl_divergence",
    title="Discrete Spatial Profile KL Divergence ($D_{\\mathrm{KL}}$)",
    ax=ax2,
    cmap="plasma_r",
    fmt=".2e",
)

fig.tight_layout()
plt.show()

## 3. 2D Histogram of Error Distributions vs. Bitwidths with Marginals

Next, we aggregate the sample-level errors across all configurations and plot the **2D histogram (`hist2d`)** of error distributions as a function of fractional bits and integer bits, with marginal distributions on the top and right axes.

In [ ]:
# Load sample-wise differences for all bit configurations
int_bits = analysis.integer_bits_range.as_list()
frac_bits = analysis.fractional_bits_range.as_list()

diff_dfs = {}
sample_records = []

for ib in int_bits:
    for fb in frac_bits:
        diff_df = analysis.get_diff_df(ib, fb).collect()
        diff_dfs[(ib, fb)] = diff_df
        sample_records.append(
            diff_df.select([
                "id",
                "sample_mae",
                "sample_mse",
                "sample_mape",
                "sample_max_error",
                "sample_kl_divergence",
            ]).with_columns([
                pl.lit(ib).alias("integer_bits"),
                pl.lit(fb).alias("fractional_bits"),
                pl.lit(ib + fb).alias("total_bits"),
            ])
        )

all_samples_diff_df = pl.concat(sample_records)

# 2D Histogram: Fractional Bits vs Sample MAE with marginals
fig, main_ax, top_ax, right_ax = plot_joint_hist2d_with_marginals(
    data=all_samples_diff_df,
    x_col="fractional_bits",
    y_col="sample_mae",
    xlabel="Fractional Bits ($fb$)",
    ylabel="Sample Mean Absolute Error (MAE)",
    title="2D Error Distribution vs. Fractional Bits with Marginal Projections",
    yscale="log",
    bins=(len(frac_bits), 40),
    cmap="Blues",
    cbar_label="Number of Events",
)
plt.show()

In [ ]:
# 2D Histogram: Total Bits vs Sample KL Divergence with marginals
unique_total_bits = sorted(all_samples_diff_df["total_bits"].unique().to_list())

fig, main_ax, top_ax, right_ax = plot_joint_hist2d_with_marginals(
    data=all_samples_diff_df,
    x_col="total_bits",
    y_col="sample_kl_divergence",
    xlabel="Total Bits ($ib + fb$)",
    ylabel="Sample Spatial KL Divergence ($D_{\\mathrm{KL}}$)",
    title="2D KL Divergence vs. Total Bitwidth with Marginal Projections",
    yscale="log",
    bins=(len(unique_total_bits), 40),
    cmap="Purples",
    marginal_color="purple",
    cbar_label="Number of Events",
)
plt.show()

## 4. Marginal Distributions for Each Variable / Ring Feature

We examine the joint distribution between unquantized normalized ring values ($y_{ij}$) and quantization absolute differences ($|y_{ij} - \hat{y}_{ij}|$) across all detector rings, including the top marginal (energy density) and right marginal (error density).

In [ ]:
# Select a specific bitwidth configuration to inspect feature-level distribution
selected_ib = int_bits[0]   # e.g., ib=1
selected_fb = frac_bits[2]  # e.g., fb=6

diff_selected = diff_dfs[(selected_ib, selected_fb)]
norm_data = analysis.get_data()

# Extract normalized columns
ring_selector = RingSlicesPerLayer(
    rings_col=analysis.rings_col,
    fraction=analysis.ring_fraction,
    output_format="expanded_columns",
)
sliced_df = norm_data.pipe(ring_selector, passthrough=True)
normalizer = AlternativeNorm1(input_cols=ring_selector.output_cols)
norm_df = sliced_df.pipe(normalizer, passthrough=True).collect()

norm_cols = normalizer.output_cols
true_vals = norm_df.select(norm_cols).to_numpy().flatten()
abs_diff_cols = [f"{c}.abs_diff" for c in norm_cols]
abs_diff_vals = diff_selected.select(abs_diff_cols).to_numpy().flatten()

# 2D Histogram: True Ring Feature Value vs Absolute Quantization Difference
fig, main_ax, top_ax, right_ax = plot_joint_hist2d_with_marginals(
    x=true_vals,
    y=abs_diff_vals,
    xlabel="True Normalized Ring Feature ($y_{ij}$)",
    ylabel="Quantization Absolute Difference ($|y_{ij} - \\hat{y}_{ij}|$)",
    title=f"Feature Joint Error Distribution ($ib={selected_ib}, fb={selected_fb}$)",
    yscale="log",
    bins=50,
    cmap="viridis",
    marginal_color="teal",
    cbar_label="Ring Observations",
)
plt.show()

## 5. Marginal Error Distribution Comparisons Across Bitwidths

Here we compare the 1D marginal error distributions:
1. **Fixed Integer Bits ($ib$)**: Observe error shifting toward zero as fractional bits ($fb$) increase.
2. **Fixed Fractional Bits ($fb$)**: Observe the impact of saturation / overflow across integer bits ($ib$).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

# 1. Vary fractional bits with fixed integer bits
fixed_ib = int_bits[0]
plot_error_distributions_by_bits(
    diff_dfs=diff_dfs,
    metric_col="sample_mae",
    fixed_bit_type="integer_bits",
    fixed_bit_val=fixed_ib,
    xlabel="Sample MAE",
    title=f"Marginal Error Distribution across $fb$ (Fixed $ib={fixed_ib}$)",
    ax=ax1,
    bins=40,
    log_x=True,
    log_y=True,
)

# 2. Vary integer bits with fixed fractional bits
fixed_fb = frac_bits[2]
plot_error_distributions_by_bits(
    diff_dfs=diff_dfs,
    metric_col="sample_mae",
    fixed_bit_type="fractional_bits",
    fixed_bit_val=fixed_fb,
    xlabel="Sample MAE",
    title=f"Marginal Error Distribution across $ib$ (Fixed $fb={fixed_fb}$)",
    ax=ax2,
    bins=40,
    log_x=True,
    log_y=True,
)

fig.tight_layout()
plt.show()

## 6. Ring-Wise Spatial Quantization Error Profile

We evaluate how the Mean Absolute Error varies across the individual ring channels ($0$ to $99$) for different bit budgets.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ring_indices = np.arange(len(norm_cols))

tested_fbs = [frac_bits[0], frac_bits[len(frac_bits)//3], frac_bits[2*len(frac_bits)//3], frac_bits[-1]]

for fb_val in tested_fbs:
    df_cfg = diff_dfs[(int_bits[0], fb_val)]
    ring_maes = [float(df_cfg[f"{col}.abs_diff"].mean()) for col in norm_cols]
    ax.plot(ring_indices, ring_maes, marker="o", markersize=3, label=f"ib={int_bits[0]}, fb={fb_val}")

ax.set_xlabel("Ring Index", fontsize="medium")
ax.set_ylabel("Mean Absolute Error per Ring", fontsize="medium")
ax.set_yscale("log")
ax.set_title("Ring-Wise Spatial Quantization Error Profile", fontsize="large")
ax.grid(linestyle="--", alpha=0.2, color="k")
ax.legend(fontsize="small", loc="best")
plt.show()

## 7. Results Summary Table

Below is the complete summary table ordered by MAE:

In [ ]:
summary_df = results_df.sort("mae")
summary_df.select([
    "integer_bits",
    "fractional_bits",
    "total_bits",
    "mae",
    "mse",
    "rmse",
    "mape",
    "max_error",
    "kl_divergence",
    "hist_kl_divergence",
])